# Engine Dispatch Verification — EURUSD April 2024

Proves NautilusTrader's two-layer bar dispatch model end-to-end for two strategy subscription patterns (5-min and 15-min INTERNAL aggregation off the same 1-min EXTERNAL source).

1. The `BacktestEngine` internal data stream (`engine.data`) holds raw 1-min EXTERNAL bars before `run()`.
2. The `DataEngine` dispatches every cached bar at runtime to subscribed strategies.
3. The `TimeBarAggregator` (one per `INTERNAL@...-EXTERNAL` subscription) consumes those dispatched EXTERNAL bars and emits aggregated bars to `on_bar`.

Six CSVs across two variants:

| Variant | CSV | Source | What it represents |
|---|---|---|---|
| **5-min** | `csv/raw_engine_data_1min_2024_04.csv` | `engine.data` **before** `run()`, filtered to 1-MIN-EXTERNAL | what the engine **holds** pre-dispatch |
| **5-min** | `csv/sniffer_engine_1min_2024_04.csv` | runtime sniffer, `aggregation_source==EXTERNAL` | bars the `DataEngine` **dispatched** |
| **5-min** | `csv/sniffer_strategy_5min_2024_04.csv` | runtime sniffer, `aggregation_source==INTERNAL` | 5-min bars the **aggregator emitted** to `on_bar` |
| **15-min** | `csv/raw_engine_data_15min_2024_04.csv` | `engine15.data` **before** `run()`, filtered to 1-MIN-EXTERNAL | what the engine **holds** pre-dispatch |
| **15-min** | `csv/sniffer_engine_15min_2024_04.csv` | runtime sniffer, `aggregation_source==EXTERNAL` | bars the `DataEngine` **dispatched** |
| **15-min** | `csv/sniffer_strategy_15min_2024_04.csv` | runtime sniffer, `aggregation_source==INTERNAL` | 15-min bars the **aggregator emitted** to `on_bar` |

All six share the same 12 columns: `stream, price_type, bar_type, ts_event_ns, ts_init_ns, open, high, low, close, volume, ts_event, ts_init`.

Per-variant assertion (key = `(bar_type, ts_init_ns)`): `set(raw_engine) == set(sniffer_engine)`  →  `missing=0  extra=0`.

In [ ]:
import csv
import sys
from decimal import Decimal
from pathlib import Path

import pandas as pd


def _find_project_root() -> Path:
    marker = Path("core") / "instrument_factory.py"

    def _walk_up(p: Path) -> Path | None:
        for parent in (p, *p.parents):
            if (parent / marker).is_file():
                return parent
        return None

    found = _walk_up(Path.cwd())
    if found:
        return found
    for var in ("__vsc_ipynb_file__", "__session__", "__file__"):
        nb = globals().get(var)
        if nb:
            found = _walk_up(Path(nb).resolve().parent)
            if found:
                return found
    for child in Path.cwd().iterdir():
        if child.is_dir() and (child / marker).is_file():
            return child
    raise RuntimeError(f"could not find project root (no `{marker}`) from cwd={Path.cwd()}")


PROJECT_ROOT = _find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CATALOG_PATH = str(PROJECT_ROOT / "catalog")
CSV_DIR = PROJECT_ROOT / "csv"
CSV_DIR.mkdir(exist_ok=True)

INSTRUMENT_ID_STR = "EURUSD.FOREX_MS"
START = "2024-04-01"
END = "2024-04-30"

# 5-min variant
CSV_RAW_ENGINE      = CSV_DIR / "raw_engine_data_1min_2024_04.csv"
CSV_SNIFFER_ENGINE  = CSV_DIR / "sniffer_engine_1min_2024_04.csv"
CSV_SNIFFER_STRAT   = CSV_DIR / "sniffer_strategy_5min_2024_04.csv"

# 15-min variant
CSV_RAW_ENGINE_15     = CSV_DIR / "raw_engine_data_15min_2024_04.csv"
CSV_SNIFFER_ENGINE_15 = CSV_DIR / "sniffer_engine_15min_2024_04.csv"
CSV_SNIFFER_STRAT_15  = CSV_DIR / "sniffer_strategy_15min_2024_04.csv"

print(f"project root: {PROJECT_ROOT}")
print(f"catalog     : {CATALOG_PATH}")
print(f"csv dir     : {CSV_DIR}")

In [ ]:
from nautilus_trader.backtest.engine import BacktestEngine
from nautilus_trader.cache.config import CacheConfig
from nautilus_trader.config import BacktestEngineConfig, LoggingConfig, RiskEngineConfig, StrategyConfig
from nautilus_trader.model import TraderId
from nautilus_trader.model.currencies import USD
from nautilus_trader.model.data import Bar, BarType
from nautilus_trader.model.enums import AccountType, AggregationSource, OmsType
from nautilus_trader.model.identifiers import InstrumentId
from nautilus_trader.model.objects import Money
from nautilus_trader.persistence.catalog.parquet import ParquetDataCatalog
from nautilus_trader.trading.strategy import Strategy

from core.instrument_factory import create_instrument

## Load 1-min BID + ASK bars from the catalog

Mirrors `core.backtest_runner._cached_catalog_bars` — same `+1d -1ns` end-of-day expansion so April 30 is fully included. The same `all_bars` list feeds both the 5-min and 15-min variant engines below.

In [ ]:
BAR_TYPE_1M_BID = f"{INSTRUMENT_ID_STR}-1-MINUTE-BID-EXTERNAL"
BAR_TYPE_1M_ASK = f"{INSTRUMENT_ID_STR}-1-MINUTE-ASK-EXTERNAL"

catalog = ParquetDataCatalog(CATALOG_PATH)
start_ts = pd.Timestamp(START, tz="UTC")
end_ts   = pd.Timestamp(END,   tz="UTC") + pd.Timedelta(days=1) - pd.Timedelta(nanoseconds=1)

bars_bid = catalog.bars(bar_types=[BAR_TYPE_1M_BID], start=start_ts, end=end_ts)
bars_ask = catalog.bars(bar_types=[BAR_TYPE_1M_ASK], start=start_ts, end=end_ts)
all_bars = list(bars_bid) + list(bars_ask)

print(f"BID bars: {len(bars_bid):,}")
print(f"ASK bars: {len(bars_ask):,}")
print(f"total   : {len(all_bars):,}")

## Shared helpers and sniffer class

Single 12-column row helper used by every CSV in the notebook. `BarSniffer` buckets bars by `aggregation_source` so one strategy class works for both the 5-min and the 15-min variant — only the subscribed `BarType` tuple changes.

In [ ]:
CSV_FIELDS = [
    "stream", "price_type", "bar_type",
    "ts_event_ns", "ts_init_ns",
    "open", "high", "low", "close", "volume",
    "ts_event", "ts_init",
]


def bar_to_row(stream: str, bar: Bar) -> dict:
    bt = bar.bar_type
    return {
        "stream":      stream,
        "price_type":  bt.spec.price_type.name,
        "bar_type":    str(bt),
        "ts_event_ns": int(bar.ts_event),
        "ts_init_ns":  int(bar.ts_init),
        "open":        float(bar.open),
        "high":        float(bar.high),
        "low":         float(bar.low),
        "close":       float(bar.close),
        "volume":      float(bar.volume),
        "ts_event":    pd.Timestamp(bar.ts_event, unit="ns", tz="UTC").isoformat(),
        "ts_init":     pd.Timestamp(bar.ts_init,  unit="ns", tz="UTC").isoformat(),
    }


def write_rows(path, rows):
    with open(path, "w", newline="", encoding="utf-8") as fh:
        w = csv.DictWriter(fh, fieldnames=CSV_FIELDS)
        w.writeheader()
        w.writerows(rows)


def build_engine(trader_id: str) -> BacktestEngine:
    eng = BacktestEngine(config=BacktestEngineConfig(
        trader_id=TraderId(trader_id),
        logging=LoggingConfig(bypass_logging=True),
        risk_engine=RiskEngineConfig(bypass=True),
        run_analysis=False,
        cache=CacheConfig(bar_capacity=200_000, drop_instruments_on_reset=False),
    ))
    eng.add_venue(
        venue=instrument.id.venue,
        oms_type=OmsType.NETTING,
        account_type=AccountType.MARGIN,
        starting_balances=[Money(1_000_000, USD)],
        base_currency=USD,
        default_leverage=Decimal(1),
    )
    eng.add_instrument(instrument)
    eng.add_data(all_bars)
    return eng


def dump_raw_engine(eng: BacktestEngine, csv_path) -> int:
    """Read engine.data (pre-run), filter to EXTERNAL Bars, write 12-col CSV."""
    rows = [
        bar_to_row("raw_engine", d)
        for d in eng.data
        if isinstance(d, Bar) and d.bar_type.aggregation_source == AggregationSource.EXTERNAL
    ]
    write_rows(csv_path, rows)
    return len(rows)


class BarSnifferConfig(StrategyConfig, frozen=True):
    instrument_id: InstrumentId
    bar_types: tuple[BarType, ...]
    csv_engine_path: str
    csv_strategy_path: str


class BarSniffer(Strategy):
    """Records every on_bar; buckets EXTERNAL vs INTERNAL; writes 2 CSVs in on_stop."""

    def __init__(self, config: BarSnifferConfig) -> None:
        super().__init__(config)
        self._engine_rows: list[dict] = []
        self._strategy_rows: list[dict] = []

    def on_start(self) -> None:
        for bt in sorted(self.config.bar_types, key=lambda b: b.aggregation_source.value):
            self.subscribe_bars(bt)

    def on_bar(self, bar: Bar) -> None:
        if bar.bar_type.aggregation_source == AggregationSource.EXTERNAL:
            self._engine_rows.append(bar_to_row("sniffer_engine", bar))
        else:
            self._strategy_rows.append(bar_to_row("sniffer_strategy", bar))

    def on_stop(self) -> None:
        write_rows(self.config.csv_engine_path,   self._engine_rows)
        write_rows(self.config.csv_strategy_path, self._strategy_rows)


instrument = create_instrument("EUR", "USD", venue="FOREX_MS")
print(f"instrument: {instrument.id}")

## Variant 1 — 5-minute aggregation

In [ ]:
engine = build_engine("DISPATCH-VERIFY-5M")
n_raw = dump_raw_engine(engine, CSV_RAW_ENGINE)
print(f"raw engine.data → {CSV_RAW_ENGINE.name}  rows={n_raw:,}")

sniffer = BarSniffer(BarSnifferConfig(
    instrument_id=instrument.id,
    bar_types=(
        BarType.from_str(f"{INSTRUMENT_ID_STR}-1-MINUTE-BID-EXTERNAL"),
        BarType.from_str(f"{INSTRUMENT_ID_STR}-1-MINUTE-ASK-EXTERNAL"),
        BarType.from_str(f"{INSTRUMENT_ID_STR}-5-MINUTE-BID-INTERNAL@1-MINUTE-EXTERNAL"),
        BarType.from_str(f"{INSTRUMENT_ID_STR}-5-MINUTE-ASK-INTERNAL@1-MINUTE-EXTERNAL"),
    ),
    csv_engine_path=str(CSV_SNIFFER_ENGINE),
    csv_strategy_path=str(CSV_SNIFFER_STRAT),
))
engine.add_strategy(sniffer)
engine.run()
print(
    f"sniffer captured: engine={len(sniffer._engine_rows):,} "
    f"strategy={len(sniffer._strategy_rows):,}"
)
engine.dispose()

In [ ]:
df1 = pd.read_csv(CSV_RAW_ENGINE)
df2 = pd.read_csv(CSV_SNIFFER_ENGINE)
df3 = pd.read_csv(CSV_SNIFFER_STRAT)

key = ["bar_type", "ts_init_ns"]
s1 = set(map(tuple, df1[key].itertuples(index=False, name=None)))
s2 = set(map(tuple, df2[key].itertuples(index=False, name=None)))
missing = s1 - s2
extra   = s2 - s1

print(f"CSV raw_engine    (1-min EXT): {len(df1):>7,} rows")
print(f"CSV sniffer_engine(1-min EXT): {len(df2):>7,} rows")
print(f"CSV sniffer_strat (5-min INT): {len(df3):>7,} rows")
print(f"missing={len(missing)}  extra={len(extra)}")
assert len(missing) == 0, f"engine dropped bars: {list(missing)[:3]}"
assert len(extra)   == 0, f"engine invented bars: {list(extra)[:3]}"
print(f"5-MIN INTERNAL per side: {df3.groupby('price_type').size().to_dict()}")

## Variant 2 — 15-minute aggregation

Fresh `BacktestEngine` with the same `all_bars` source. The strategy now subscribes to `15-MINUTE-{BID,ASK}-INTERNAL@1-MINUTE-EXTERNAL`. `engine.data` and the dispatched feed both still contain **1-MIN-EXTERNAL** bars — the only thing that changes is what the `TimeBarAggregator` produces (15-min instead of 5-min).

Expected 15-min `sniffer_strategy` count per side: 30 days × 96 windows/day = **2,880** (clock-driven aggregator, one bar per wall-clock boundary regardless of input gaps).

In [ ]:
engine15 = build_engine("DISPATCH-VERIFY-15M")
n_raw15 = dump_raw_engine(engine15, CSV_RAW_ENGINE_15)
print(f"raw engine15.data → {CSV_RAW_ENGINE_15.name}  rows={n_raw15:,}")

sniffer15 = BarSniffer(BarSnifferConfig(
    instrument_id=instrument.id,
    bar_types=(
        BarType.from_str(f"{INSTRUMENT_ID_STR}-1-MINUTE-BID-EXTERNAL"),
        BarType.from_str(f"{INSTRUMENT_ID_STR}-1-MINUTE-ASK-EXTERNAL"),
        BarType.from_str(f"{INSTRUMENT_ID_STR}-15-MINUTE-BID-INTERNAL@1-MINUTE-EXTERNAL"),
        BarType.from_str(f"{INSTRUMENT_ID_STR}-15-MINUTE-ASK-INTERNAL@1-MINUTE-EXTERNAL"),
    ),
    csv_engine_path=str(CSV_SNIFFER_ENGINE_15),
    csv_strategy_path=str(CSV_SNIFFER_STRAT_15),
))
engine15.add_strategy(sniffer15)
engine15.run()
print(
    f"sniffer15 captured: engine={len(sniffer15._engine_rows):,} "
    f"strategy={len(sniffer15._strategy_rows):,}"
)
engine15.dispose()

In [ ]:
df1_15 = pd.read_csv(CSV_RAW_ENGINE_15)
df2_15 = pd.read_csv(CSV_SNIFFER_ENGINE_15)
df3_15 = pd.read_csv(CSV_SNIFFER_STRAT_15)

s1_15 = set(map(tuple, df1_15[key].itertuples(index=False, name=None)))
s2_15 = set(map(tuple, df2_15[key].itertuples(index=False, name=None)))
missing_15 = s1_15 - s2_15
extra_15   = s2_15 - s1_15

print(f"CSV raw_engine_15     (1-min EXT): {len(df1_15):>7,} rows")
print(f"CSV sniffer_engine_15 (1-min EXT): {len(df2_15):>7,} rows")
print(f"CSV sniffer_strat_15  (15-min INT): {len(df3_15):>6,} rows")
print(f"missing={len(missing_15)}  extra={len(extra_15)}")
assert len(missing_15) == 0, f"engine dropped bars: {list(missing_15)[:3]}"
assert len(extra_15)   == 0, f"engine invented bars: {list(extra_15)[:3]}"
print(f"15-MIN INTERNAL per side: {df3_15.groupby('price_type').size().to_dict()}")

# What did each layer hold, in plain English?
print()
print("=== summary: what's in each layer for the 15-min variant ===")
print(f"engine.data        held: {sorted(df1_15['bar_type'].unique())}")
print(f"DataEngine dispatched : {sorted(df2_15['bar_type'].unique())}")
print(f"strategy on_bar got   : {sorted(df3_15['bar_type'].unique())}")